In [ ]:
import pandas as pd
df = pd.read_csv("E:\\data center - water electricity\\data_center_hybrid.csv")
print(df.head())
print(df.info())
print(df.columns)
print(df.describe())

In [ ]:
print("facility names :")
print(df['Facility_Name'].unique())
print(len(df['Facility_Name'].unique()))

print("cities :")
print(df['City'].unique())
print(len(df['City'].unique()))

print("country :")
print(df['Country'].unique())
print(len(df['Country'].unique()))

print('Facility_Type :')
print(df['Facility_Type'].unique())
print(len(df['Facility_Type'].unique()))

print('Owner_Company :')
print(df['Owner_Company'].unique())
print(len(df['Owner_Company'].unique()))

print('Year :')
print(df['Year'].unique())
print(len(df['Year'].unique()))

'Cooling_System_Type'
print('Cooling_System_Type :')
print(df['Cooling_System_Type'].unique())
print(len(df['Cooling_System_Type'].unique()))


In [ ]:
# countries list is showing 344 unique values. some countries are seen repeated(different langauges).
for country in sorted(df["Country"].dropna().unique()):
    print(country)

In [ ]:
import re
import unicodedata
import pycountry
import country_converter as coco

In [ ]:


# Official ISO country names
valid_countries = {c.name for c in pycountry.countries}

# Alias dictionary
aliases = {
    "Unknown": None,
    "Serbia and Montenegro": "Serbia",
    "Netherlands Antilles": "Netherlands",
    "Myanmar (Burma)": "Myanmar",
    "USA": "United States",
    "US": "United States",
    "UK": "United Kingdom",
    "Russia": "Russian Federation",
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Vietnam": "Viet Nam",
    "Czech Republic": "Czechia",
    "Cape Verde": "Cabo Verde",
    "Ivory Coast": "Côte d'Ivoire",
    "Swaziland": "Eswatini",
    "Macedonia": "North Macedonia",
    "Republic of Korea": "Korea, Republic of",
    "España": "Spain",
    "Canadá": "Canada",
    "México": "Mexico",
    "Malasia": "Malaysia",
    "Chipre": "Cyprus",
    "Grecia": "Greece",
    "Turquía": "Türkiye",
    "Tailandia": "Thailand",
    "Países Bajos": "Netherlands",
    "República Dominicana": "Dominican Republic",
    "Japón": "Japan",
    "Sudáfrica": "South Africa",
    "Taiwán": "Taiwan, Province of China",
    "Bolivia": "Bolivia, Plurinational State of",
    "Bélgica": "Belgium",
    "Moldávia": "Moldova, Republic of",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "România": "Romania",
    "Rumanía": "Romania",
    "Việt Nam": "Viet Nam",
    "Suomi": "Finland",
    "Sverige": "Sweden",
    "Norge": "Norway",
    "Noorwegen": "Norway",
    "Danmark": "Denmark",
    "Svizzera": "Switzerland",
    "Finlandia": "Finland",
    "Germania": "Germany",
    "Allemagne": "Germany",
    "États-Unis": "United States",
    "Royaume-Uni": "United Kingdom",
    "Italie": "Italy",
    "Espagne": "Spain",
    "La Réunion": "Réunion",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "Россия": "Russian Federation",
    "България": "Bulgaria",
    "Polska": "Poland",
    "香港": "Hong Kong",
    "Congo": "Republic of the Congo",
    "Republic of the Congo": "Republic of the Congo",
    "Democratic Republic of the Congo": "Congo, Democratic People's Republic of",
    "Alemanha": "Germany",
    "Estados Unidos": "United States",
    "Reino Unido": "United Kingdom",
    "Itália": "Italy",
    "França": "France",
    "Suíça": "Switzerland",
    "Polónia": "Poland",
    "Áustria": "Austria",
    "Holanda": "Netherlands",
    "Moçambique": "Mozambique",
    "Brasil": "Brazil",
    "Nederland": "Netherlands",
    "Niederlande": "Netherlands",
    "Duitsland": "Germany",
    "Österreich": "Austria",
    "Schweiz": "Switzerland",
    "België": "Belgium",
    "Iran": "Iran, Islamic Republic of",
    "Tanzania": "Tanzania, United Republic of",
    "Taiwan": "Taiwan, Province of China",
    "IN": "India",
    "Maharashtra": "India",
    "Turkey": "Türkiye",
    "The Gambia": "Gambia"
}
aliases.update({
    "Congo": "Republic of the Congo",
    "Republic of the Congo": "Republic of the Congo",
    "Democratic Republic of the Congo": "Congo, Democratic People's Republic of",
    "Moldova": "Moldova, Republic of",
    "Moldávia": "Moldova, Republic of",
    
   
})


def clean_country(text):
    if text is None:
        return None

    text = str(text).strip()
    text = unicodedata.normalize("NFKC", text)

    # Remove postal codes
    text = re.sub(r"\s+\d+$", "", text)

    # Special cases
    text = re.sub(r"(Taiwan)\s+\d+$", r"Taiwan", text)
    text = re.sub(r"(Taiwán)\s+\d+$", r"Taiwán", text)
    text = re.sub(r"(Maharashtra)\s+\d+$", r"Maharashtra", text)

    # Ignore pure numbers
    if text.isdigit():
        return None

    # Remove address-like entries
    if any(word in text.lower() for word in [
        "chome","road","street","building","apple store","ビル","郵政编码","号",
        "株式会社","有限会社","公司","corporation","centre","center"
    ]):
        return None
    # Apply aliases first (case-insensitive)
    alias_lower = {k.lower(): v for k,v in aliases.items()}
    alias_key = text.lower()
    if alias_key in alias_lower:
        mapped = alias_lower[alias_key]
        if mapped is None:
            return None
        text = mapped
    # coco conversion
    converted = coco.convert(names=text, to="name_short", not_found=None)
    if converted:
        text = converted

    # apply aliases (case-insensitive)
    alias_key = text.lower()
    alias_lower = {k.lower(): v for k,v in aliases.items()}
    if alias_key in alias_lower:
        mapped = alias_lower[alias_key]
        if mapped is None:
            return None
        text = mapped

    # Extra normalization
    extra = {
        "Vietnam": "Viet Nam",
        "Russia": "Russian Federation",
        "South Korea": "Korea, Republic of",
        "North Korea": "Korea, Democratic People's Republic of",
        "Ivory Coast": "Côte d'Ivoire",
        "Cape Verde": "Cabo Verde",
        "Czech Republic": "Czechia",
        "Swaziland": "Eswatini",
    }
    text = extra.get(text, text)

    return text if text in valid_countries else None

# Apply cleaning
values = df["Country"].dropna().astype(str).str.strip().unique()

cleaned, rejected = [], []

for item in values:
    country = clean_country(item)
    if country:
        cleaned.append(country)
    else:
        rejected.append(item)

cleaned = sorted(set(cleaned))
rejected = sorted(set(rejected))

print("Countries found", len(cleaned))
print(cleaned)

print("\nRejected values", len(rejected))
for r in rejected:
    print(r)


In [ ]:
# rows that were rejected
rejected_rows = df[df["Country"].isin(rejected)]

# number of rows affected
num_rejected_rows = len(rejected_rows)

print("Number of rows affected by rejected values:", num_rejected_rows)

rejected_counts = df["Country"].value_counts().loc[rejected]
print(rejected_counts)


In [ ]:
# Drop rows where Country is in the rejected list
df_cleaned = df[~df["Country"].isin(rejected)]

print("Original rows:", len(df))
print("Rows after dropping rejected:", len(df_cleaned))
print("Rows dropped:", len(df) - len(df_cleaned))


In [ ]:
# check data in company names
for owner in sorted(df_cleaned['Owner_Company'].dropna().unique()):
    print(owner)

In [ ]:
# Normalize company names
df_cleaned["Company_normalized"] = df_cleaned['Owner_Company'].str.strip().str.lower()

# Alias dictionary for duplicates
aliases = {
    # 123NET variants
    "123 net": "123NET",
    "123net": "123NET",
    "123net": "123NET",

    # Amazon variants
    "amazon aws": "Amazon",
    "amazon": "Amazon",

    # Apple variants
    "apple inc.": "Apple",
    "apple": "Apple",

    # AT TOKYO variants
    "at tokyo corporation": "AT TOKYO",
    "at tokyo": "AT TOKYO",

    # Atlantic.Net variants
    "atlantic.net": "Atlantic.Net",
    "atlantic metro communications": "Atlantic.Net",

    # Cogent Communications variants
    "cogent communications inc.": "Cogent Communications",
    "cogent communications, inc.": "Cogent Communications",
    "cogent communications": "Cogent Communications",

    # Bezeq International variants
    "bezeq international (bi)": "Bezeq International",
    "bezeq international ltd. (bi)": "Bezeq International",

    # Hetzner variants
    "hetzner online gmbh": "Hetzner",
    "hetzner (pty) ltd": "Hetzner",
    "hetzner": "Hetzner",

    # FirstLight variants
    "firstlight fiber": "FirstLight",
    "first light": "FirstLight",
    "firstlight": "FirstLight",

    # CyrusOne variants
    "cyrusone data centers": "CyrusOne",
    "cyrusone": "CyrusOne",
}

# Apply aliases
df_cleaned["Company_cleaned"] = df_cleaned["Company_normalized"].replace(aliases)

# Identify rejected values (non-company junk, if you have a rejected list)

rejected = df_cleaned[df_cleaned["Company_cleaned"].isna()]['Owner_Company'].unique()

# Drop rejected rows
df_cleaned = df_cleaned[~df_cleaned['Owner_Company'].isin(rejected)]

print("Original rows:", len(df_cleaned))
print("Rows after dropping rejected:", len(df_cleaned))
print("Rows dropped:", len(df) - len(df_cleaned))

# Check duplicates after cleaning
duplicates = df_cleaned["Company_cleaned"].value_counts()
print("Potential duplicates after cleaning:")
print(duplicates[duplicates > 1])

In [ ]:
# Check data in column City
for cities in sorted(df_cleaned['City'].dropna().unique()):
    print(cities)

In [ ]:
# Normalize city names directly in City column
df_cleaned["City"] = df_cleaned["City"].str.strip().str.lower()

# Alias dictionary for duplicates / variants
city_aliases = {
    "bogota": "Bogotá",
    "bucharest": "București",
    "bucuresti": "București",
    "bucurești": "București",
    "krakow": "Kraków",
    "warsaw": "Warszawa",
    "warszaw": "Warszawa",
    "wroclaw": "Wrocław",
    "nuernberg": "Nürnberg",
    "nuremberg": "Nürnberg",
    "munich": "München",
    "moscow": "Москва",
    "moskva": "Москва",
    "kyiv": "Kyiv",
    "kiev": "Kyiv",
    "beograd": "Belgrade",
    "brasilia": "Brasília",
    "brasilía": "Brasília",
    "lisbon": "Lisboa",
    "milan": "Milano",
    "florence": "Firenze",
    "vienna": "Wien",
    "zurich": "Zürich",
    "geneva": "Genève",
    "athens": "Athína",
    "istanbul": "İstanbul",
    "ankara": "Ankara",
    "bangalore": "Bengaluru",
    "bombay": "Mumbai",
    "delhi": "New Delhi",
    "kolkatta": "Kolkata",
    "kolkata": "Kolkata",
    "madras": "Chennai",
    "peking": "Beijing",
    "guangzhou": "Canton",
    "saint petersburg": "Sankt-Peterburg",
    "st. petersburg": "Sankt-Peterburg",
    "saint louis": "St. Louis",
    "st louis": "St. Louis",
    "st. paul": "St. Paul",
    "saint paul": "St. Paul",
    "unknown": "Unknown City",
    "states": "United States",
}
city_aliases.update({
    "zürich": "Zurich",
    "Zürich": "Zurich",
    "çekmeköy": "Cekmekoy",
    "øvrebø": "Ovrebø",
    "istanbul": "Istanbul",
    "šempeter": "Sempeter",
    "москва": "Moscow",
    "الرياض": "Riyadh",
    "حائل": "Ha'il",
    "อำเภอเมือง": "Mueang District",
    "เขต": "Khet",
    "香港": "Hong Kong",
    "vélizy": "Velizy",
    "vélizy-villacoublay": "Velizy-Villacoublay",
    "vénissieux": "Venissieux",
    "vitória": "Vitoria",
    "veberöd": "Veberod",
    "unterschleißheim": "Unterschleissheim",
    "törökbálint": "Torokbalint",
    "poznań": "Poznan",
    "tønder": "Tonder",
    "põhja-tallinna": "Pohja-Tallinna",
    "uberlândia": "Uberlandia",
    "timișoara": "Timisoara",
    "são": "Sao",
    "spånga": "Spanga",
    "sodankylä": "Sodankyla",
    "setúbal": "Setubal",
    "saint-ouen-l'aumône": "Saint-Ouen",
    "rümlang": "Rumlang",
    "querétaro": "Queretaro",
    "rüsselsheim": "Russelsheim",
    "podgórne": "Podgorne",
    "nyíregyháza": "Nyiregyhaza",
    "nørre": "Norre",
    "nørresundby": "Norresundby",
    "nürnberg": "Nurnberg",
    "niš": "Nis",
    "new york": "New York",
    "måløy": "Maloy",
    "mérignac": "Merignac",
    "lørenskog": "Lorenskog",
    "mörfelden": "Morfelden",
    "logroño": "Logrono",
    "münchen": "Munich",
    "liège": "Liege",
    "münchenstein": "Munchenstein",
    "münsbach": "Munsbach",
    "münster": "Munster",
    "maastricht": "Maastricht",
    "luleå": "Lulea",
    "labège": "Labege",
    "kraków": "Krakow",
    "kemiönsaari": "Kemionsaari",
    "jundiaí": "Jundiai",
    "järfälla": "Jarvalla",
    "hortolândia": "Hortolandia",
    "jõhvi": "Johvi",
    "hägersten": "Hagersten",
    "huixquilucan,": "Huixquilucan",
    "hünenbert": "Hunenbert",
    "hamburg-fuhlsbüttel": "Hamburg",
    "grödinge": "Grodinge",
    "gdańsk": "Gdansk",
    "frölunda": "Frolunda",
    "eyüp": "Eyup",
    "düdingen": "Dudingen",
    "dalton, ga": "Dalton",
    "düsseldorf": "Dusseldorf",
    "cârcea": "Carcea",
    "covilhã": "Covilha",
    "bélem": "Belem",
    "budaörs": "Budaors",
    "brno-střed-pisárky": "Brno",
    "bakırköy": "Bakirkoy",
    "bahçelievler": "Bahcelievler",
    "arnavutköy": "Arnavutkoy",
    "alcalá": "Alcala",
    "wrocław": "Wroclaw",
    "'s-hertogenbosch": "Hertogenbosch",
    "brasília": "Brasilia",
    "bucurești": "Bucharest",
    "zurich": "Zurich",
    "zürich": "Zurich",
    "bogotá": "Bogota",
    "rd." : "rd",
    "new York city" : "New York"
})

# Apply aliases directly to City column
df_cleaned["City"] = df_cleaned["City"].replace(city_aliases)

In [ ]:


def handle_numeric(city):
    if city is None:
        return None
    city = str(city).strip()

    # Explicitly catch values starting with #
    if city.startswith("#"):
        return "Numeric_Code"

    # Pure numbers
    if city.isdigit():
        return "Numeric_Code"

    # Phone-like strings (digits, +, -, .)
    if re.match(r"^[\d\-\.\+\']+$", city):
        return "Numeric_Code"

    # Any string containing digits anywhere
    if re.search(r"\d", city):
        return "Numeric_Code"

    # Emails
    if re.match(r"[^@]+@[^@]+\.[^@]+", city):
        return "Unknown_City"

    # URLs
    if city.startswith("www.") or city.startswith("http"):
        return "Unknown_City"

    # Otherwise return the original city
    return city

# Apply to DataFrame
df_cleaned["City"] = df_cleaned["City"].replace(city_aliases)
df_cleaned["City"] = df_cleaned["City"].apply(handle_numeric)

# Check counts
print(df_cleaned["City"].value_counts().head(20))


In [ ]:
for cities in sorted(df_cleaned['City'].dropna().unique()):
    print(cities)
    print(len(cities))

In [ ]:
for facility in sorted(df_cleaned['Facility_Name'].dropna().unique()):
    print(facility)
   

In [ ]:
# remove odd characters but keep () / - for addresses
df_cleaned["Facility_Name"] = (
    df_cleaned["Facility_Name"]
    .str.lower()
    .str.strip()
    .str.replace(r"[^\w\s\(\)\./-]", "", regex=True) 
)


In [ ]:
for facility in sorted(df_cleaned['Facility_Name'].dropna().unique()):
    print(facility)

In [ ]:
print(df_cleaned.info())
print(df_cleaned.head(20))

In [ ]:
# Drop extra columns
df_cleaned = df_cleaned.drop(columns=["Company_normalized", "Company_cleaned"])

# Verify remaining columns
print(df_cleaned.columns)




In [ ]:
# Save cleaned DataFrame to CSV
df_cleaned.to_csv("cleaned_facilities.csv", index=False)


In [ ]:
print(df_cleaned.shape)
print(df_cleaned.dtypes)
print(df_cleaned.isnull().sum())


PUE — Power Usage Effectiveness
Definition: Ratio of total facility energy consumption to IT equipment energy consumption.
A PUE of 1.0 means all energy goes directly to computing (perfect efficiency, no overhead).
Typical modern data centers achieve 1.2–1.5.
Higher values (>2.0) indicate significant overhead from cooling, lighting, or other systems.

WUE — Water Usage Effectiveness
Definition: Measures water consumption relative to IT energy usage.
Lower WUE means less water is consumed for cooling per unit of computing work.
Important for facilities in water‑stressed regions, where sustainability is critical.

In [ ]:
print(df_cleaned.describe(include="all"))


In [ ]:
print(df_cleaned["Owner_Company"].value_counts().head(10))
print(df_cleaned["Country"].value_counts().head(10))
print(df_cleaned["City"].value_counts().head(10))


In [ ]:
import matplotlib.pyplot as plt

df_cleaned["PUE"].hist(bins=20)
plt.title("Distribution of PUE")
plt.show()

df_cleaned["WUE_L_per_kWh"].hist(bins=20)
plt.title("Distribution of WUE")
plt.show()

print(df["Surrounding_Water_Stress_Tier"].value_counts())


Distribution of PUE (Power Usage Effectiveness)
Most values cluster around 1.6 → this means that for every 1 watt used by servers, ~0.6 watts are spent on cooling, lighting, and other overhead.
Range 1.4–1.8 dominates → the bulk of facilities are moderately efficient, but not cutting‑edge.
Few facilities below 1.2 → very few data centers achieve near‑optimal efficiency.
Tail toward 2.0 → some sites are still highly inefficient, consuming almost as much overhead as IT load.

Interpretation: The industry average is still far from ideal (1.0), showing that cooling and infrastructure overhead remain a major energy burden.

Distribution of WUE (Water Usage Effectiveness)

Strong skew toward 0–0.5 → most facilities use very little water per kWh, often due to air or liquid cooling adoption.
Smaller distribution between 1.0–3.0 → these are evaporative‑cooled sites, consuming far more water.
Peak near 0.1–0.3 → indicates a growing trend toward low‑water cooling strategies.
Long tail up to 3.0 → some facilities are extremely water‑intensive, likely in hot/dry climates with evaporative cooling.

Interpretation: While many data centers are moving toward water‑efficient cooling, a significant minority still rely on evaporative systems, which are unsustainable in water‑stressed regions.

In [ ]:
print(df_cleaned.groupby("Facility_Type")["Estimated_Capacity_MW"].describe())


In [ ]:
print(df_cleaned.groupby("Cooling_System_Type")["WUE_L_per_kWh"].describe())


Air cooled vs liquid cooled → both are water‑efficient, but liquid cooling is slightly better.

Evaporative → orders of magnitude higher water use, making it unsustainable in water‑stressed regions.

Facility counts → show industry inertia: most sites are still air cooled, while hyperscale campuses lean evaporative, and liquid cooling is emerging but niche.

In [ ]:
import seaborn as sns

sns.heatmap(df_cleaned[["Estimated_Capacity_MW","Daily_Electricity_Usage_MWh",
                "Daily_Water_Usage_Gallons","PUE","WUE_L_per_kWh"]].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


Strong Positive Correlations

Estimated Capacity MW ↔ Daily Electricity Usage MWh → 0.98
Nearly perfect correlation. Larger capacity facilities consume proportionally more electricity.
This is expected: IT load scales directly with capacity.

Daily Water Usage Gallons ↔ Daily Electricity Usage MWh → strong positive (~0.9).
Facilities that use more electricity also tend to use more water
Indicates resource intensity scales together.


Negative Correlations

PUE ↔ Estimated Capacity MW → -0.6
Larger facilities tend to have lower PUE (better energy efficiency).
Hyperscale operators benefit from economies of scale and advanced cooling designs.

WUE L/kWh ↔ Estimated Capacity MW → moderate negative.
Bigger sites often adopt air or liquid cooling, reducing water intensity per kWh compared to smaller evaporative‑cooled sites.

In [ ]:
# Top countries by daily electricity usage
top_electricity = (
    df_cleaned.groupby("Country")["Daily_Electricity_Usage_MWh"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print("Top Countries by Daily Electricity Usage (MWh):")
print(top_electricity)

# Top countries by daily water usage
top_water = (
    df_cleaned.groupby("Country")["Daily_Water_Usage_Gallons"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print("\nTop Countries by Daily Water Usage (Gallons):")
print(top_water)


In [ ]:
# Aggregate daily water usage by city
city_water_usage = (
    df_cleaned.groupby(["City", "Surrounding_Water_Stress_Tier"])["Daily_Water_Usage_Gallons"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("Top Cities by Daily Water Usage (with Water Stress Tier):")
print(city_water_usage)


In [ ]:
# Aggregate daily water usage by company owner
top_company_water = (
    df_cleaned.groupby("Owner_Company")["Daily_Water_Usage_Gallons"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("Top Company Owners by Daily Water Usage (Gallons):")
print(top_company_water)


Amazon AWS

Daily water usage: ~4.0 billion gallons
By far the largest consumer of water.
Driven by widespread use of evaporative cooling in hyperscale campuses.

Google

Daily water usage: ~1.58 billion gallons
Heavy reliance on evaporative cooling at sites like Eemshaven and Fredericia, each consuming tens of millions of gallons daily.

Microsoft

Daily water usage: ~1.35 billion gallons
Lower than Google despite similar scale, thanks to liquid cooling adoption at some hyperscale sites.

Equinix & Digital Realty

Usage: ~1.0B and ~0.6B gallons respectively.
As colocation providers, their footprint is spread across many smaller facilities, but water demand is still substantial.

Meta

Daily water usage: ~662M gallons.

Hyperscale campuses often use evaporative cooling, driving high water demand.

Vantage Data Centers

Daily water usage: ~333M gallons.
Smaller footprint compared to hyperscale operators, but still significant.

In [ ]:
# Count facilities per company
top_companies_by_facilities = (
    df_cleaned.groupby("Owner_Company")["Facility_Name"]
    .count()
    .reset_index(name="facility_count")
    .sort_values("facility_count", ascending=False)
    .head(10)
)

print("Top 10 Companies by Number of Data Centers:")
print(top_companies_by_facilities)


In [ ]:
# Top 10 companies by facility count
top10_companies = [
    "Amazon AWS", "Equinix", "Lumen", "China Telecom", 
    "Zenlayer", "Digital Realty", "DataBank", 
    "Google", "Microsoft", "Digital Realty Trust"
]

# Aggregate electricity, water, and capacity for these companies
consumption_capacity_by_company = (
    df[df["Owner_Company"].isin(top10_companies)]
    .groupby("Owner_Company")
    .agg(
        facility_count=("Facility_Name", "count"),
        avg_capacity=("Estimated_Capacity_MW", "mean"),
        avg_electricity=("Daily_Electricity_Usage_MWh", "mean"),
        avg_water=("Daily_Water_Usage_Gallons", "mean")
    )
    .reset_index()
    .sort_values("facility_count", ascending=False)
)

print("Top 10 Companies by Facility Count with Capacity, Electricity & Water Consumption:")
print(consumption_capacity_by_company)


In [ ]:
# List of target companies
companies = [
    "Amazon AWS", "Google", "Equinix", "Microsoft", 
    "Amazon", "Meta", "Microsoft Azure", 
    "Digital Realty", "Vantage Data Centers"
]

# Group by company and cooling system, aggregate counts + consumption
cooling_by_company = (
    df[df["Owner_Company"].isin(companies)]
    .groupby(["Owner_Company", "Cooling_System_Type"])
    .agg(
        facility_count=("Facility_Name", "count"),
        avg_electricity=("Daily_Electricity_Usage_MWh", "mean"),
        avg_water=("Daily_Water_Usage_Gallons", "mean")
    )
    .reset_index()
    .sort_values(["Owner_Company", "facility_count"], ascending=[True, False])
)

print("Cooling System Counts + Electricity & Water Consumption by Company:")
print(cooling_by_company)



Cooling System Counts + Consumption Insights

Amazon (non‑AWS)
Evaporative (574 facilities) → avg electricity ~3,452 MWh/day, avg water ~1.54M gallons/day.
Air cooled (126) → similar electricity (~3,452 MWh/day), but much lower water (~118k gallons/day).
Liquid cooled (84) → slightly lower electricity (~3,184 MWh/day), lowest water (~92k gallons/day).

--> Amazon’s non‑AWS footprint is smaller, but evaporative sites are still water‑intensive.

Amazon AWS

Evaporative (2,597) → avg electricity ~3,457 MWh/day, avg water ~1.49M gallons/day.
Air cooled (784) → avg electricity ~3,264 MWh/day, avg water ~113k gallons/day.
Liquid cooled (350) → avg electricity ~3,820 MWh/day, avg water ~87k gallons/day.

--> AWS dominates globally. Evaporative cooling drives massive water demand, while liquid cooling shows better water efficiency but slightly higher electricity.

Google

Evaporative (1,036) → avg electricity ~3,422 MWh/day, avg water ~1.48M gallons/day.
Air cooled (252) → avg electricity ~3,547 MWh/day, avg water ~140k gallons/day.
Liquid cooled (147) → avg electricity ~3,290 MWh/day, avg water ~69k gallons/day.

--> Google’s largest sites (Eemshaven, Fredericia) skew water usage upward. Liquid cooling is far more sustainable.

Microsoft

Evaporative (875) → avg electricity ~3,421 MWh/day, avg water ~1.49M gallons/day.
Air cooled (273) → avg electricity ~3,696 MWh/day, avg water ~146k gallons/day.
Liquid cooled (168) → avg electricity ~3,721 MWh/day, avg water ~93k gallons/day.

--> Microsoft is actively shifting toward liquid cooling, reducing water footprint while keeping electricity comparable.

Microsoft Azure

Evaporative (413) → avg electricity ~3,415 MWh/day, avg water ~1.48M gallons/day.
Air cooled (140) → avg electricity ~3,458 MWh/day, avg water ~133k gallons/day.
Liquid cooled (77) → avg electricity ~3,284 MWh/day, avg water ~69k gallons/day.


Equinix

Evaporative (1,946) → avg electricity ~1,418 MWh/day, avg water ~484k gallons/day.
Air cooled (1,211) → avg electricity ~1,481 MWh/day, avg water ~42k gallons/day.
Liquid cooled (217) → avg electricity ~1,356 MWh/day, avg water ~23k gallons/day.

As a colocation provider, Equinix has lower electricity averages per facility, but evaporative sites still consume huge water volumes.

Digital Realty

Evaporative (1,127) → avg electricity ~1,441 MWh/day, avg water ~513k gallons/day.
Air cooled (889) → avg electricity ~1,343 MWh/day, avg water ~40k gallons/day.
Liquid cooled (77) → avg electricity ~1,673 MWh/day, avg water ~33k gallons/day.

Similar to Equinix: colocation footprint, but evaporative cooling drives water demand.

Meta

Evaporative (441) → avg electricity ~3,329 MWh/day, avg water ~1.45M gallons/day.
Air cooled (147) → avg electricity ~3,549 MWh/day, avg water ~124k gallons/day.
Liquid cooled (56) → avg electricity ~4,792 MWh/day, avg water ~79k gallons/day.

Meta’s liquid cooled sites show excellent water efficiency, but electricity averages are higher — likely due to AI/HPC workloads.

Vantage Data Centers

Evaporative (609) → avg electricity ~1,399 MWh/day, avg water ~510k gallons/day.
Air cooled (427) → avg electricity ~1,450 MWh/day, avg water ~49k gallons/day.
Liquid cooled (84) → avg electricity ~1,649 MWh/day, avg water ~21k gallons/day.

Smaller footprint, but same pattern: evaporative = water‑intensive, liquid = sustainable.

Cross‑Company Insights
Evaporative cooling → consistently ~1.4–1.5M gallons/day per facility for hyperscale operators (AWS, Google, Microsoft, Meta).
Air cooling → ~100–150k gallons/day, but electricity averages are slightly higher.
Liquid cooling → lowest water usage (~20k–90k gallons/day), electricity averages vary but are comparable or slightly higher.

AWS & Google → most exposed to water sustainability risks due to heavy evaporative reliance.
Microsoft & Meta → actively shifting toward liquid cooling, showing better water efficiency.
Equinix & Digital Realty → colocation providers with lower per‑facility averages, but still significant water demand overall.

Liquid cooling is the most sustainable path forward, especially for hyperscale and AI workloads.
AWS and Google face the largest sustainability risks due to evaporative cooling dominance.
Microsoft is positioning themselves as leaders by adopting liquid cooling.

Colocation providers (Equinix, Digital Realty) have lower averages but still contribute heavily due to scale.

In [ ]:
# Aggregate daily electricity usage by company owner
top_company_electricity = (
    df.groupby("Owner_Company")["Daily_Electricity_Usage_MWh"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("Top Company Owners by Daily Electricity Usage (MWh):")
print(top_company_electricity)


Amazon AWS

Daily electricity usage: 12,872,450 MWh
Far ahead of all other operators — AWS dominates global cloud infrastructure.
Reflects their massive hyperscale footprint worldwide.

Google

Daily usage: 4,922,627 MWh
Significant consumption, but less than half of AWS.
Google has invested heavily in renewable energy offsets.

Equinix & Digital Realty

Usage: ~4.8M and ~2.9M MWh respectively.
Both are colocation giants, hosting multiple tenants.
Their electricity footprint reflects broad global presence.

Microsoft & Microsoft Azure

Combined usage: ~6.7M MWh.
Shows Microsoft’s dual reporting (corporate vs Azure cloud).
Strong focus on liquid cooling adoption for efficiency.

Meta

Usage: ~2.25M MWh.
Heavy demand driven by social platforms and AI workloads.
Meta has been building hyperscale campuses with renewable energy integration.

Vantage Data Centers

Usage: ~1.6M MWh.
Smaller footprint compared to hyperscale operators, but growing rapidly.

AWS is the clear leader in electricity consumption, reflecting its dominance in cloud services.Google and Microsoft are strong but significantly behind AWS in raw consumption.Colocation providers (Equinix, Digital Realty) still consume massive electricity due to multi‑tenant hosting.Cooling strategies and efficiency investments (liquid vs evaporative) will play a huge role in reducing these footprints.

In [ ]:
# Group by cooling system type and calculate averages + counts
cooling_usage = (
    df_cleaned.groupby("Cooling_System_Type")
    .agg(
        avg_electricity=("Daily_Electricity_Usage_MWh", "mean"),
        avg_water=("Daily_Water_Usage_Gallons", "mean"),
        facility_count=("Cooling_System_Type", "count")
    )
    .sort_values("avg_electricity", ascending=False)
)

print("Cooling System vs Average Electricity & Water Usage (with Facility Count):")
print(cooling_usage)


Air Cooled

Average electricity: ~404 MWh/day (lowest among the three).
Average water: ~11,666 gallons/day (moderate).
Facility count: ~79,541 (most common cooling type).
Air cooling is the most widely deployed, but it tends to be less efficient electrically compared to liquid cooling. It saves water compared to evaporative systems.

Evaporative Cooling

Average electricity: ~883 MWh/day (moderate).
Average water: ~342,348 gallons/day (extremely high).
Facility count: ~45,612.
Evaporative cooling is water‑intensive, consuming ~30x more water than air cooling. It’s chosen for energy efficiency but is unsustainable in water‑stressed regions.

Liquid Cooling

Average electricity: ~2,655 MWh/day (highest).
Average water: ~56,542 gallons/day (lowest).
Facility count: ~1,617 (least common).
Liquid cooling is rare but growing, especially for high‑density workloads (AI, HPC). It consumes more electricity but dramatically reduces water usage compared to evaporative cooling.

In [ ]:
# Top facilities by estimated capacity (MW) with electricity, water usage, and cooling type
top_capacity = (
    df.groupby(["Facility_Name", "Owner_Company", "Cooling_System_Type"])[
        ["Estimated_Capacity_MW", "Daily_Electricity_Usage_MWh", "Daily_Water_Usage_Gallons"]
    ]
    .sum()
    .sort_values("Estimated_Capacity_MW", ascending=False)
    .head(10)
)

print("Top Facilities by Estimated Capacity (MW) with Electricity, Water Usage, and Cooling Type:")
print(top_capacity)


Scale of Facilities
The largest facilities (Google Eemshaven, Google Fredericia, Google Southeast Nebraska, Microsoft Plano, etc.) have capacities above 2,000 MW, which places them firmly in the hyperscale category.

These hyperscale sites are consuming tens of thousands of MWh daily — for example, Google Eemshaven uses ~67,900 MWh/day.

Water Consumption Patterns
Evaporative cooling facilities (Google Eemshaven, Fredericia, Nebraska; Microsoft Plano; AWS Wade Dr; AWS Atlantic Blvd) show extremely high water usage — tens of millions of gallons per day.

Example: Google Eemshaven → 35 million gallons/day.

Microsoft Plano → 29 million gallons/day.

Liquid cooled facilities (Microsoft Gaines Township, Google Botetourt, AWS CMH Beech) use dramatically less water — often under 1 million gallons/day.

Example: Google Botetourt → only ~100,000 gallons/day despite ~2,200 MW capacity.

Air cooled facilities (AWS CMH Hayden) also show low water usage (~800,000 gallons/day), but electricity consumption remains high.

Across all cooling types, electricity usage is very high (40,000–67,000 MWh/day).

Liquid cooled facilities (e.g., Microsoft Gaines Township, AWS CMH Beech) consume similar electricity to evaporative ones, but with far lower water usage.

Air cooled facilities (AWS CMH Hayden) consume ~46,500 MWh/day, comparable to liquid cooling, but with moderate water usage.

Google’s hyperscale evaporative sites (Eemshaven, Fredericia, Nebraska) are water‑intensive giants, raising sustainability concerns if located in water‑stressed regions.

Microsoft and AWS liquid cooled sites demonstrate much lower water footprints, suggesting a strategic shift toward sustainable cooling.

Air cooled sites (like AWS CMH Hayden) still consume high electricity, showing a trade‑off where water is saved but energy efficiency suffers.

Overall, cooling choice is the biggest driver of water usage differences among hyperscale facilities — electricity consumption remains consistently massive across all types.

Amazon AWS CMH – 5109 Hayden shows lower water and electricity consumption compared to AWS – 44150 Wade Dr and AWS IAD – 21195 Atlantic Blvd, even though its estimated capacity is higher.Facilities may operate below peak capacity depending on demand, redundancy, or staged expansion.

Amazon AWS CMH – 5109 Hayden may not be running at full load, while Wade Dr and Atlantic Blvd could be closer to their operational maximum.

In [ ]:


# Filter facilities in high water-stress areas
water_stressed_df = df_cleaned[df_cleaned["Surrounding_Water_Stress_Tier"] == "High"]

# Aggregate by company
consumption_in_stressed = (
    water_stressed_df.groupby("Owner_Company")
    .agg(
        facility_count=("Facility_Name", "count"),
        avg_water=("Daily_Water_Usage_Gallons", "mean"),
        avg_electricity=("Daily_Electricity_Usage_MWh", "mean"),
        total_capacity=("Estimated_Capacity_MW", "sum"),
    )
    .reset_index()
    .sort_values("avg_water", ascending=False)
)

print("Water Consumption & Facility Counts in High Water-Stressed Areas:")
print(consumption_in_stressed.head(10))


AWS dominates in both facility count and total capacity in water‑stressed areas, making it the most vulnerable to water scarcity.
Google and Microsoft have similar footprints, but Microsoft’s balanced cooling mix reduces average water consumption slightly.
Meta shows the trade‑off: lower water use with liquid cooling, but higher electricity intensity.
Apple and Alibaba have smaller footprints, but their per‑facility averages are very high, suggesting less efficient cooling strategies.
DC 12 14 DE LLC (Amazon) is an outlier with nearly 2M gallons/day per facility, highlighting extreme water intensity.

In [ ]:


# Filter only high water-stress areas
water_stressed_df = df_cleaned[df_cleaned["Surrounding_Water_Stress_Tier"] == "High"]

# Aggregate by company
consumption_in_stressed = (
    water_stressed_df.groupby("Owner_Company")
    .agg(
        facility_count=("Facility_Name", "count"),
        avg_water=("Daily_Water_Usage_Gallons", "mean"),
        avg_electricity=("Daily_Electricity_Usage_MWh", "mean"),
        total_capacity=("Estimated_Capacity_MW", "sum")
    )
    .reset_index()
)

# Calculate intensity metrics
consumption_in_stressed["water_per_MW"] = (
    consumption_in_stressed["avg_water"] / consumption_in_stressed["total_capacity"]
)
consumption_in_stressed["electricity_per_MW"] = (
    consumption_in_stressed["avg_electricity"] / consumption_in_stressed["total_capacity"]
)

# Step 3: Sort by water intensity (highest to lowest)
efficiency_metrics = consumption_in_stressed.sort_values("water_per_MW", ascending=False)

print("Efficiency Metrics (Water & Electricity per MW of Capacity in High Water-Stressed Areas):")
print(efficiency_metrics.head(10))


In [ ]:


# Define hyperscale operators of interest
hyperscale_companies = [
    "Amazon AWS", "Amazon", "Google", "Microsoft", 
    "Meta", "Microsoft Azure", "Apple", "Apple Inc.", "Alibaba Group"
]

# Filter only high water-stress areas for these companies
hyperscale_df = df_cleaned[
    (df_cleaned["Surrounding_Water_Stress_Tier"] == "High") &
    (df_cleaned["Owner_Company"].isin(hyperscale_companies))
]

# Aggregate by company
hyperscale_metrics = (
    hyperscale_df.groupby("Owner_Company")
    .agg(
        facility_count=("Facility_Name", "count"),
        avg_water=("Daily_Water_Usage_Gallons", "mean"),
        avg_electricity=("Daily_Electricity_Usage_MWh", "mean"),
        total_capacity=("Estimated_Capacity_MW", "sum")
    )
    .reset_index()
)

# Calculate efficiency metrics
hyperscale_metrics["water_per_MW"] = (
    hyperscale_metrics["avg_water"] / hyperscale_metrics["total_capacity"]
)
hyperscale_metrics["electricity_per_MW"] = (
    hyperscale_metrics["avg_electricity"] / hyperscale_metrics["total_capacity"]
)

# Sort by water intensity
hyperscale_metrics = hyperscale_metrics.sort_values("water_per_MW", ascending=False)

print("Hyperscale Operators Efficiency Metrics (Water & Electricity per MW in High Water-Stressed Areas):")
print(hyperscale_metrics)

In [ ]:

# Group by year and facility type
yearly_consumption = (
    df_cleaned.groupby(["Year", "Facility_Type"])
    .agg(
        total_water=("Daily_Water_Usage_Gallons", "sum"),
        total_electricity=("Daily_Electricity_Usage_MWh", "sum")
    )
    .reset_index()
)

# Pivot for easier plotting
pivot_water = yearly_consumption.pivot(index="Year", columns="Facility_Type", values="total_water").fillna(0)
pivot_electricity = yearly_consumption.pivot(index="Year", columns="Facility_Type", values="total_electricity").fillna(0)

# Plot water consumption
pivot_water.plot(kind="line", marker="o", figsize=(12,6))
plt.ylabel("Total Water Consumption (Gallons/day)")
plt.title("Year-wise Water Consumption by Facility Type")
plt.legend(title="Facility Type")
plt.tight_layout()
plt.show()

# Plot electricity consumption
pivot_electricity.plot(kind="line", marker="o", figsize=(12,6))
plt.ylabel("Total Electricity Consumption (MWh/day)")
plt.title("Year-wise Electricity Consumption by Facility Type")
plt.legend(title="Facility Type")
plt.tight_layout()
plt.show()


Enterprise/Standard → stable, low growth in both water and electricity.

Colocation → moderate growth, steady but not disruptive.

Hyperscale/AI → exponential growth in both water and electricity, driving the industry’s sustainability debate.

The resource intensity of hyperscale AI workloads is reshaping the environmental footprint of data centers worldwide.

In [ ]:


# Group by Year and Country
yearly_country_consumption = (
    df_cleaned.groupby(["Year", "Country"])
    .agg(
        total_water=("Daily_Water_Usage_Gallons", "sum"),
        total_electricity=("Daily_Electricity_Usage_MWh", "sum")
    )
    .reset_index()
)

# Select top 10 countries by overall water consumption
top10_countries = (
    yearly_country_consumption.groupby("Country")["total_water"].sum()
    .nlargest(10)
    .index
)

filtered = yearly_country_consumption[yearly_country_consumption["Country"].isin(top10_countries)]

# Pivot for plotting
pivot_water = filtered.pivot(index="Year", columns="Country", values="total_water").fillna(0)
pivot_electricity = filtered.pivot(index="Year", columns="Country", values="total_electricity").fillna(0)

# Plot water consumption
pivot_water.plot(kind="line", marker="o", figsize=(12,6))
plt.ylabel("Total Water Consumption (Gallons/day)")
plt.title("Year wise Water Consumption by Top 10 Countries")
plt.legend(title="Country")
plt.tight_layout()
plt.show()

# Plot electricity consumption
pivot_electricity.plot(kind="line", marker="o", figsize=(12,6))
plt.ylabel("Total Electricity Consumption (MWh/day)")
plt.title("Year-wise Electricity  Consumption by Top 10 Countries")

In [ ]:


#  Group by Year and Country
yearly_country_consumption = (
    df_cleaned.groupby(["Year", "Country"])
    .agg(
        total_water=("Daily_Water_Usage_Gallons", "sum"),
        total_electricity=("Daily_Electricity_Usage_MWh", "sum")
    )
    .reset_index()
)

# Select top 10 countries by overall water consumption (excluding United States)
top10_countries = (
    yearly_country_consumption[yearly_country_consumption["Country"] != "United States"]
    .groupby("Country")["total_water"].sum()
    .nlargest(10)
    .index
)

# Filter dataset
filtered = yearly_country_consumption[
    yearly_country_consumption["Country"].isin(top10_countries)
]

# Pivot for plotting
pivot_water = filtered.pivot(index="Year", columns="Country", values="total_water").fillna(0)
pivot_electricity = filtered.pivot(index="Year", columns="Country", values="total_electricity").fillna(0)

# Plot water consumption
pivot_water.plot(kind="line", marker="o", figsize=(12,6))
plt.ylabel("Total Water Consumption (Gallons/day)")
plt.title("Year wise Water Consumption by Top 10 Countries (Excluding US)")
plt.legend(title="Country")
plt.tight_layout()
plt.show()

# Plot electricity consumption
pivot_electricity.plot(kind="line", marker="o", figsize=(12,6))
plt.ylabel("Total Electricity Consumption (MWh/day)")
plt.title("Year wise Electricity Consumption by Top 10 Countries (Excluding US)")
plt.legend(title="Country")
plt.tight_layout()
plt.show()


Water Consumption (Top 10 Countries)

-United States dominates: In the full chart, US water consumption dwarfs all others, rising from ~1.2 billion gallons/day in 2019 to ~2.1 billion by 2025.

-Excluding US: Australia emerges as the largest consumer among the rest, with sharp growth from ~20 million to ~80 million gallons/day.

-Other countries (China, India, Germany, UK, Canada, Japan, Ireland, Netherlands, Brazil) show steady but much smaller increases.

Cumulative insight: The US alone accounts for the majority of global water demand, but smaller nations like Australia and Ireland are growing disproportionately fast relative to their size.

Electricity Consumption (Top 10 Countries)
-United States again dominates: Rising from ~5 million MWh/day in 2019 to nearly 9 million by 2025.

-Excluding US: The UK leads, climbing from ~300,000 to ~350,000 MWh/day. China and Germany also show steady growth, while Brazil and Ireland remain lower but upward trending.

Cumulative insight: The US is the clear driver of electricity demand, but Europe (UK, Germany, Netherlands) and Asia (China, India, Japan) are steadily expanding their footprints.

